### CSV and Excel Files- Structured Data

In [1]:
import pandas as pd
import os

In [2]:
os.makedirs("data/structured_files",exist_ok=True)

In [3]:
data = {
    "Product": ["Laptop", "Wireless Mouse", "Monitor", "Keyboard"],
    "Price": [999.99, 29.99, 179.99, 49.99],
    "Stock": [15, 120, 45, 80],
    "Description": [
        "15.6-inch display with 16GB RAM and 512GB SSD",
        "Ergonomic 2.4GHz optical mouse with USB receiver",
        "27-inch Full HD IPS monitor with 75Hz refresh rate",
        "Tenkeyless mechanical keyboard with RGB backlighting",
    ],
}

df=pd.DataFrame(data)
df.to_csv("data/structured_files/products.csv",index=False)

In [4]:
#Save as Excel with multiple sheets
with pd.ExcelWriter('data/structured_files/inventory.xlsx') as writer:
    df.to_excel(writer,sheet_name="Products",index=False)

    #add another sheet
    summary_data={
        "Category":["Electronics","Acessories"],
        "Total_items":[3,2],
        "Total_value":[1389.97,109.98]
    }
    pd.DataFrame(summary_data).to_excel(writer,sheet_name="Summary",index=False)

### CSV Processing

In [5]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader

C:\Users\mihir\AppData\Local\Temp\ipykernel_23644\444435838.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader


In [6]:
#Method-1: CSVLoader -Each Row Becomes a Document
print("CSVLoader-Row based Documents")
csv_loader=CSVLoader(
    file_path="data/structured_files/products.csv",
    encoding="utf-8",
    csv_args={
        "delimiter":",",
        "quotechar":'"',
    }
)
csv_docs=csv_loader.load()
print(f"Loaded {len(csv_docs)} documents (one per row)")
print("\nFrist Document:")
print(f"Content: {csv_docs[0].page_content}")
print(f"Metadata: {csv_docs[0].metadata}")

CSVLoader-Row based Documents
Loaded 4 documents (one per row)

Frist Document:
Content: Product: Laptop
Price: 999.99
Stock: 15
Description: 15.6-inch display with 16GB RAM and 512GB SSD
Metadata: {'source': 'data/structured_files/products.csv', 'row': 0}


In [7]:
# Method-2 CUSTOM CSV PROCESSING

from importlib import metadata

from langchain_core.documents import Document
from typing import List

print("Custom CSV Processing")
def process_csv_intelligently(filepath:str)->List[Document]:
    """Process CSV with intelligent document creation"""
    df=pd.read_csv(filepath)
    documents=[]

    #Strategy 1: One document per row with structured content
    for idx, row in df.iterrows():
        # Create structured content
        content=f"""Product Information:
        Name: {row["Product"]}
        Price: ${row["Price"]}
        Stock: {row["Stock"]} units
        Description: {row["Description"]}"""

        # Create document with rich metadata
        doc=Document(
           page_content=content,
           metadata={
               "source":filepath,
               "row_index": idx,
               "product_name":row["Product"],
               "price":row["Price"],
               "data_type": "product_info"
           } 
        )
        documents.append(doc)
    return documents

Custom CSV Processing


In [8]:
process_csv_intelligently("data/structured_files/products.csv")

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'price': 999.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Laptop\n        Price: $999.99\n        Stock: 15 units\n        Description: 15.6-inch display with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': 'Wireless Mouse', 'price': 29.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Wireless Mouse\n        Price: $29.99\n        Stock: 120 units\n        Description: Ergonomic 2.4GHz optical mouse with USB receiver'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product_name': 'Monitor', 'price': 179.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Monitor\n        Price: $179.99\n        Stock: 45 units\n        Description: 27-inch Full HD IPS monitor

### Excel Processing

In [9]:
# Method-1: Using pandas for full control
print("1 Pandas-based excel processing")
def process_excel_with_pandas(filepath: str)-> List[Document]:
    """Process Excel with sheet awareness"""
    documents=[]

    #Read all Sheets
    excel_file=pd.ExcelFile(filepath)

    for sheet_name in excel_file.sheet_names:
        df=pd.read_excel(filepath,sheet_name=sheet_name)

        #create document for each sheet
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content +=df.to_string(index=False)

        doc=Document(
            page_content=sheet_content,
            metadata={
                "source": filepath,
                "sheet_name": sheet_name,
                "num_rows": len(df),
                "num_columns": len(df.columns),
                "data_type": "excel_sheet"
            }
        )
        documents.append(doc)
    return documents

1 Pandas-based excel processing


In [12]:
excel_docs=process_excel_with_pandas(r"data\structured_files/inventory.xlsx")
excel_docs

[Document(metadata={'source': 'data\\structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 4, 'num_columns': 4, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\nColumns: Product, Price, Stock, Description\nRows: 4\n\n       Product  Price  Stock                                          Description\n        Laptop 999.99     15        15.6-inch display with 16GB RAM and 512GB SSD\nWireless Mouse  29.99    120     Ergonomic 2.4GHz optical mouse with USB receiver\n       Monitor 179.99     45   27-inch Full HD IPS monitor with 75Hz refresh rate\n      Keyboard  49.99     80 Tenkeyless mechanical keyboard with RGB backlighting'),
 Document(metadata={'source': 'data\\structured_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}, page_content='Sheet: Summary\nColumns: Category, Total_items, Total_value\nRows: 2\n\n   Category  Total_items  Total_value\nElectronics            3      1389.97\n Acessories    

In [22]:
from langchain_community.document_loaders import UnstructuredExcelLoader
# method-2 unstructuredExcelLoader

try:
    excel_loader=UnstructuredExcelLoader(
        r"data\structured_files/inventory.xlsx",
        mode="elements"
    )
    unstructured_docs=excel_loader.load()

except Exception as e:
    print(f"Error :{e}")

In [23]:
unstructured_docs

[Document(metadata={'source': 'data\\structured_files/inventory.xlsx', 'file_directory': 'data\\structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-09-14T13:14:23', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>999.99</td><td>15</td><td>15.6-inch display with 16GB RAM and 512GB SSD</td></tr><tr><td>Wireless Mouse</td><td>29.99</td><td>120</td><td>Ergonomic 2.4GHz optical mouse with USB receiver</td></tr><tr><td>Monitor</td><td>179.99</td><td>45</td><td>27-inch Full HD IPS monitor with 75Hz refresh rate</td></tr><tr><td>Keyboard</td><td>49.99</td><td>80</td><td>Tenkeyless mechanical keyboard with RGB backlighting</td></tr></table>', 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', 'category': 'Table', 'element_id': '8c06335a6f1ececc58d342b78a8f535a'}, page_content='Product Price Stock Description La